In [106]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [107]:
!pip install lightgbm
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from metric import precision_at_recall

1. **Загрузка данных**

In [108]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


2. **EDA**

In [109]:
print(train['target'].value_counts(normalize=True)) #сильный дисбаланс классов, В модели используем class_weight='balanced' или scale_pos_weight.

target
0    0.918943
1    0.081057
Name: proportion, dtype: float64


***Типы событий***

In [110]:
print(events['event_name'].value_counts())
print()
print(events['eid'].value_counts())

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

eid
200    120817
100    100402
210     36517
400     19049
220     17403
300     11316
900      7928
500      6267
301      6125
303      3081
Name: count, dtype: int64


***Пропуски***

In [111]:
print(events.isna().mean().round(3).sort_values(ascending=False))

search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
seller_type      0.407
item_id          0.348
item_category    0.100
item_location    0.072
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
dtype: float64


In [112]:
print("Сырые значения platform")
print(events['platform'].value_counts().head(15))

# Нормализация
def normalize_platform(p):
    if pd.isna(p):
        return 'unknown'
    p = str(p).lower()
    if 'android' in p:
        return 'android'
    if 'ios' in p or 'iphone' in p or 'ipad' in p:
        return 'ios'
    if 'desktop' in p:
        return 'desktop'
    if 'web' in p:
        return 'web'
    return 'other'

events['platform_norm'] = events['platform'].apply(normalize_platform)
print("\nПосле нормализации")
print(events['platform_norm'].value_counts())

Сырые значения platform
platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

После нормализации
platform_norm
web        138792
android    126875
desktop     46866
ios         16372
Name: count, dtype: int64


***User-Agent — явные боты***

In [113]:
# Ищем явные признаки ботов в UA
ua_lower = events['user_agent'].fillna('').str.lower()
bot_markers = ['headless', 'python', 'curl', 'wget', 'bot', 'spider', 'scrapy', 'http']
for marker in bot_markers:
    count = ua_lower.str.contains(marker).sum()
    if count > 0:
        print(f"UA с '{marker}': {count}")

# Топ-10 UA
print("\nТоп-10 user_agent")
print(events['user_agent'].value_counts().head(10))

UA с 'headless': 12447
UA с 'python': 2165
UA с 'curl': 1209
UA с 'scrapy': 1210
UA с 'http': 51649

Топ-10 user_agent
user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36    3737
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0                                                       3670
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36     3664
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36     3640
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36     3558
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36                  3481
Mozilla/5.0 (Windows NT 10.0; 

***События на куку (до фильтрации по окну)***

In [114]:
ev_per_cookie = events.groupby('cookie_id').size()
print("Событий на куку")
print(ev_per_cookie.describe())
print(f"\nКук с 1 событием: {(ev_per_cookie == 1).sum()}")
print(f"Кук с >1000 событий: {(ev_per_cookie > 1000).sum()}")

Событий на куку
count    16000.000000
mean        20.556562
std         19.702527
min          1.000000
25%          8.000000
50%         14.000000
75%         27.000000
max        191.000000
dtype: float64

Кук с 1 событием: 239
Кук с >1000 событий: 0


***Фильтрация событий по окну***

In [115]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


***Временные паттерны***

In [116]:
ev_tr_sorted = ev_tr.sort_values(['cookie_id', 'event_ts']).copy()

# Интервалы между событиями
ev_tr_sorted['delta'] = ev_tr_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

print("Интервалы между событиями (сек)")
print(ev_tr_sorted['delta'].describe())

# Часы активности
ev_tr_sorted['hour'] = ev_tr_sorted['event_ts'].dt.hour
hours_per_cookie = ev_tr_sorted.groupby('cookie_id')['hour'].nunique()
print("\nУникальных часов на куку")
print(hours_per_cookie.describe())
print(f"Кук, активных 24 часа: {(hours_per_cookie == 24).sum()}")

Интервалы между событиями (сек)
count    187345.000000
mean        690.226721
std        1844.059318
min           0.000000
25%          16.000000
50%          40.000000
75%         106.000000
max        9760.000000
Name: delta, dtype: float64

Уникальных часов на куку
count    11091.000000
mean         3.139663
std          2.221963
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         17.000000
Name: hour, dtype: float64
Кук, активных 24 часа: 0


***Сравнение поведения ботов и людей***

In [117]:
# Мержим с target
ev_tr_labeled = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')

# Сравниваем ключевые метрики
print("Сравнение ботов и людей")

# Число событий
n_events = ev_tr_labeled.groupby(['cookie_id', 'target']).size().reset_index(name='n')
print("\nСреднее число событий:")
print(n_events.groupby('target')['n'].mean())

# Уникальных item_id
n_items = ev_tr_labeled.groupby(['cookie_id', 'target'])['item_id'].nunique().reset_index(name='n')
print("\nСреднее уникальных item_id:")
print(n_items.groupby('target')['n'].mean())

# Доля captcha
captcha = ev_tr_labeled[ev_tr_labeled['event_name'] == 'captcha_shown'].groupby('cookie_id').size()
all_cookies = train['cookie_id']
captcha_rate = (captcha.reindex(all_cookies).fillna(0) > 0).astype(int)
captcha_df = pd.DataFrame({'cookie_id': all_cookies, 'has_captcha': captcha_rate.values}).merge(train[['cookie_id', 'target']], on='cookie_id')
print("\nДоля кук с captcha по классам:")
print(captcha_df.groupby('target')['has_captcha'].mean())

# Доля login
login = ev_tr_labeled[ev_tr_labeled['event_name'] == 'login'].groupby('cookie_id').size()
login_rate = (login.reindex(all_cookies).fillna(0) > 0).astype(int)
login_df = pd.DataFrame({'cookie_id': all_cookies, 'has_login': login_rate.values}).merge(train[['cookie_id', 'target']], on='cookie_id')
print("\nДоля кук с login по классам:")
print(login_df.groupby('target')['has_login'].mean())

Сравнение ботов и людей

Среднее число событий:
target
0    16.869015
1    29.484983
Name: n, dtype: float64

Среднее уникальных item_id:
target
0     8.917190
1    13.779755
Name: n, dtype: float64

Доля кук с captcha по классам:
target
0    0.0
1    0.0
Name: has_captcha, dtype: float64

Доля кук с login по классам:
target
0    0.268838
1    0.192436
Name: has_login, dtype: float64


***КЛЮЧЕВЫЕ ВЫВОДЫ***

1. Дисбаланс: ботов ~8%. Нужен class_weight='balanced'.

2. Явные маркеры ботов в UA: headless, python, curl.
   → Добавить флаг is_bot_ua.

3. captcha_shown — сильный сигнал бота (см. выше).
   → Добавить has_captcha.

4. login — сигнал человека.
   → Добавить has_login.

5. Интервалы между событиями: у ботов минимальные.
   → Добавить dt_min, dt_std, dt_median.

6. Число уникальных часов: боты активны 24/7.
   → Добавить n_unique_hours, hour_entropy.

7. Число событий и уникальных item_id — базовые признаки.

8. Платформа и OS — нормализовать и one-hot.

3. **Feature Engineering**

In [118]:
# Нормализация платформы
def normalize_platform(p):
    if pd.isna(p):
        return 'unknown'
    p = str(p).lower()
    if 'android' in p:
        return 'android'
    if 'ios' in p or 'iphone' in p or 'ipad' in p:
        return 'ios'
    if 'desktop' in p:
        return 'desktop'
    if 'web' in p:
        return 'web'
    return 'other'

events['platform_norm'] = events['platform'].apply(normalize_platform)

# Парсинг user_agent
def parse_ua(ua):
    if pd.isna(ua):
        return 'unknown', 'unknown', 0, 0
    ua_l = str(ua).lower()
    # OS
    if 'windows' in ua_l:
        os = 'windows'
    elif 'mac os' in ua_l or 'macintosh' in ua_l:
        os = 'macos'
    elif 'android' in ua_l:
        os = 'android'
    elif 'iphone' in ua_l or 'ipad' in ua_l or 'ios' in ua_l:
        os = 'ios'
    elif 'linux' in ua_l:
        os = 'linux'
    else:
        os = 'other'
    # Browser
    if 'chrome' in ua_l and 'edg' not in ua_l:
        browser = 'chrome'
    elif 'firefox' in ua_l:
        browser = 'firefox'
    elif 'safari' in ua_l and 'chrome' not in ua_l:
        browser = 'safari'
    elif 'edg' in ua_l:
        browser = 'edge'
    elif 'opera' in ua_l or 'opr' in ua_l:
        browser = 'opera'
    else:
        browser = 'other'
    # Флаги
    is_mobile = int(('mobile' in ua_l) or ('android' in ua_l) or ('iphone' in ua_l))
    is_headless = int('headless' in ua_l or 'python' in ua_l or 'curl' in ua_l or 'wget' in ua_l)
    return os, browser, is_mobile, is_headless

ua_parsed = events['user_agent'].apply(lambda x: pd.Series(parse_ua(x), index=['os', 'browser', 'is_mobile', 'is_headless']))
events = pd.concat([events, ua_parsed], axis=1)

print("Готово. Колонки:", events.columns.tolist())

Готово. Колонки: ['cookie_id', 'event_ts', 'eid', 'event_name', 'platform', 'user_agent', 'item_id', 'item_category', 'item_location', 'seller_type', 'search_query', 'search_page', 'pointer_x', 'pointer_y', 'platform_norm', 'os', 'browser', 'is_mobile', 'is_headless']


In [119]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

print(f"События в окне train: {len(ev_tr)}")
print(f"События в окне test: {len(ev_te)}")

События в окне train: 198436
События в окне test: 89690


In [120]:
# Расширенная история: все события до конца окна
def events_before_window_end(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_end_ts']], on='cookie_id')
    return ev[ev.event_ts < ev.window_end_ts]

ev_tr_all = events_before_window_end(events, train)
ev_te_all = events_before_window_end(events, test)

# Считаем флаги по всей истории
def add_history_flags(feats, ev_all):
    for name in ['login', 'contact_phone_show', 'contact_chat_open', 
                 'contact_message_sent', 'favorite_add', 'seller_page_view', 'photo_swipe']:
        has = ev_all[ev_all['event_name'] == name].groupby('cookie_id').size() > 0
        feats[f'has_{name}_hist'] = has.reindex(feats.index).fillna(False).astype(int)
    return feats

In [121]:
def build_features(ev, ev_all, meta):
    """
    ev: события внутри окна (с platform_norm, os, browser, is_mobile, is_headless)
    ev_all: все события до конца окна (для исторических флагов)
    meta: train или test (cookie_id + окна)
    """
    g = ev.groupby('cookie_id')
    feats = pd.DataFrame(index=g.size().index)
    
    # --- Базовые агрегаты ---
    feats['n_events'] = g.size()
    feats['n_unique_items'] = g['item_id'].nunique()
    feats['n_unique_categories'] = g['item_category'].nunique()
    feats['n_unique_locations'] = g['item_location'].nunique()
    feats['n_unique_sellers'] = g['seller_type'].nunique()
    feats['n_unique_queries'] = g['search_query'].nunique()
    feats['n_unique_eids'] = g['eid'].nunique()
    feats['n_unique_ua'] = g['user_agent'].nunique()
    
    # --- Временные ---
    ev_sorted = ev.sort_values(['cookie_id', 'event_ts']).copy()
    ev_sorted['delta'] = ev_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    delta_stats = ev_sorted.groupby('cookie_id')['delta'].agg(['mean', 'std', 'min', 'max', 'median'])
    delta_stats.columns = ['dt_mean', 'dt_std', 'dt_min', 'dt_max', 'dt_median']
    feats = feats.join(delta_stats)
    
    ev_sorted['hour'] = ev_sorted['event_ts'].dt.hour
    hour_stats = ev_sorted.groupby('cookie_id')['hour'].agg(['nunique', 'std', 'mean'])
    hour_stats.columns = ['n_unique_hours', 'hour_std', 'hour_mean']
    feats = feats.join(hour_stats)
    
    # Плотность поведения
    feats['items_per_event'] = feats['n_unique_items'] / feats['n_events']
    feats['repetition_rate'] = feats['n_events'] / (feats['n_unique_items'] + 1)
    
    # Ночная активность (боты активны 24/7)
    ev_sorted['is_night'] = ev_sorted['hour'].isin([0,1,2,3,4,5]).astype(int)
    feats['night_ratio'] = ev_sorted.groupby('cookie_id')['is_night'].mean()
    def hour_entropy(hours):
        p = hours.value_counts(normalize=True)
        return -(p * np.log(p + 1e-9)).sum()
    feats['hour_entropy'] = ev_sorted.groupby('cookie_id')['hour'].apply(hour_entropy)
    
    session = ev.groupby('cookie_id')['event_ts'].agg(['min', 'max'])
    feats['session_duration'] = (session['max'] - session['min']).dt.total_seconds()
    
    # --- Доли типов событий ---
    ev['ev_name'] = ev['event_name'].fillna('unknown')
    pivot = pd.crosstab(ev['cookie_id'], ev['ev_name'], normalize='index')
    pivot.columns = [f'ev_ratio_{c}' for c in pivot.columns]
    feats = feats.join(pivot)
    
    # --- Флаги ключевых событий (внутри окна) ---
    for name in ['captcha_shown', 'login', 'contact_phone_show', 'contact_chat_open', 
                 'contact_message_sent', 'favorite_add', 'seller_page_view', 'photo_swipe']:
        has = ev[ev['event_name'] == name].groupby('cookie_id').size() > 0
        feats[f'has_{name}'] = has.reindex(feats.index).fillna(False).astype(int)
    
    # --- Pointer ---
    ptr = ev.dropna(subset=['pointer_x', 'pointer_y'])
    if len(ptr) > 0:
        ptr_stats = ptr.groupby('cookie_id').agg(
            ptr_x_std=('pointer_x', 'std'),
            ptr_y_std=('pointer_y', 'std'),
            ptr_x_mean=('pointer_x', 'mean'),
            ptr_y_mean=('pointer_y', 'mean'),
            ptr_n=('pointer_x', 'size'),
        )
        feats = feats.join(ptr_stats)
        feats['ptr_fill_ratio'] = feats['ptr_n'] / feats['n_events']
    else:
        feats['ptr_fill_ratio'] = 0
    
    # --- Search page ---
    sp = ev.dropna(subset=['search_page'])
    if len(sp) > 0:
        sp_stats = sp.groupby('cookie_id')['search_page'].agg(['mean', 'max', 'std'])
        sp_stats.columns = ['search_page_mean', 'search_page_max', 'search_page_std']
        feats = feats.join(sp_stats)
    
    # --- Платформа ---
    plat = pd.crosstab(ev['cookie_id'], ev['platform_norm'], normalize='index')
    plat.columns = [f'plat_{c}' for c in plat.columns]
    feats = feats.join(plat)
    
    # --- OS ---
    os_pivot = pd.crosstab(ev['cookie_id'], ev['os'], normalize='index')
    os_pivot.columns = [f'os_{c}' for c in os_pivot.columns]
    feats = feats.join(os_pivot)
    
    # --- Флаги ---
    feats['is_headless'] = ev.groupby('cookie_id')['is_headless'].max()
    feats['is_mobile'] = ev.groupby('cookie_id')['is_mobile'].max()
    
    feats = feats.fillna(0)
    out = meta[['cookie_id']].merge(feats.reset_index(), on='cookie_id', how='left').fillna(0)
    return out

In [122]:
Xtr = build_features(ev_tr, ev_tr_all, train)
Xte = build_features(ev_te, ev_te_all, test)

print(f"Xtr: {Xtr.shape}, Xte: {Xte.shape}")
print(f"Признаков: {Xtr.shape[1] - 1}")

Xtr: (11091, 60), Xte: (4909, 60)
Признаков: 59


4. **Модель**

In [123]:
feature_cols = [c for c in Xtr.columns if c != 'cookie_id']
ytr = train['target'].values

# Временная валидация
is_valid = train['window_start_ts'] >= '2026-04-17'
X_train, y_train = Xtr.loc[~is_valid, feature_cols], ytr[~is_valid]
X_val, y_val = Xtr.loc[is_valid, feature_cols], ytr[is_valid]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    class_weight='balanced',
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

p_val = model.predict_proba(X_val)[:, 1]

print('P@R0.7:', round(precision_at_recall(y_val, p_val), 4))
print('PR-AUC:', round(average_precision_score(y_val, p_val), 4))
print('ROC-AUC:', round(roc_auc_score(y_val, p_val), 4))

Train: (9140, 59), Val: (1951, 59)
[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7244
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.924089	valid_0's binary_logloss: 0.204707
Early stopping, best iteration is:
[102]	valid_0's auc: 0.924449	valid_0's binary_logloss: 0.203064
P@R0.7: 0.7671
PR-AUC: 0.7447
ROC-AUC: 0.9244


In [124]:
model_full = lgb.LGBMClassifier(**model.get_params())
model_full.fit(Xtr[feature_cols], ytr)

p_test = model_full.predict_proba(Xte[feature_cols])[:, 1]

sub = pd.DataFrame({'cookie_id': Xte['cookie_id'], 'score': p_test})
assert len(sub) == len(test)
assert sub['score'].between(0, 1).all()
sub.to_csv('submission.csv', index=False)
print(sub.head())
print(f"Сохранено: {len(sub)} строк")

[LightGBM] [Info] Number of positive: 899, number of negative: 10192
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013550 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7389
[LightGBM] [Info] Number of data points in the train set: 11091, number of used features: 58
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
             cookie_id         score
0  ck_315fb710a0e371e7  5.839284e-07
1  ck_a76ee3b3e3e522fd  1.805446e-02
2  ck_94c9a4d382689e82  2.860056e-04
3  ck_8eaf9509ad9462a0  1.005569e-06
4  ck_9a88a5a989cb5bc6  1.177282e-04
Сохранено: 4909 строк
